# 04 - Conflict Correlation Analysis
===

Statistical analysis of AIS → conflict relationships.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 150, 'figure.figsize': (10, 6)})
sns.set_style('whitegrid')

DATA_DIR = Path('./data/processed')
OUTPUT_DIR = Path('./outputs/tables')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df = pd.read_parquet(DATA_DIR / 'ais_features.parquet')
print(f"Records: {len(df):,}")

## Event Study Analysis

In [ ]:
from src.analysis.correlation_analyzer import ConflictCorrelationAnalyzer

analyzer = ConflictCorrelationAnalyzer(str(OUTPUT_DIR))

# Event study for each zone
zones = df['conflict_zone_name'].unique()
zones = [z for z in zones if z != 'none']

event_results = []
for zone in zones[:3]:
    result = analyzer.event_study(df, '2022-02-24', zone, window_days=7)
    if not result.empty:
        result['zone'] = zone
        event_results.append(result)
        print(f"Zone: {zone}, Results: {len(result)}")

if event_results:
    combined = pd.concat(event_results, ignore_index=True)
    print(combined)

## Zone Statistics

In [ ]:
zone_stats = analyzer.compute_zone_statistics(df)
print(zone_stats)
zone_stats.to_csv(OUTPUT_DIR / 'zone_statistics.csv', index=False)

## Traffic Analysis

In [ ]:
from src.analysis.traffic_analyzer import TrafficAnalyzer

traffic = TrafficAnalyzer(str(OUTPUT_DIR))
results = traffic.run_all(df)
print(f"Traffic analysis complete: {len(results)} outputs")

## Summary

In [ ]:
print(f"Correlation analysis complete! Output: {OUTPUT_DIR}")